In [2]:
import os
import pandas as pd
import numpy as np
from matplotlib.ticker import PercentFormatter
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from scipy.interpolate import BSpline
from smt.sampling_methods import LHS
import pickle

### 获取基准几何数据

In [3]:
#baseline -- Blade_data
geometry_file = "APC107E_geometry.xlsx"#load the data of baseline geometry from excel
df = pd.read_excel(geometry_file,header=1)
df = df.iloc[:,:3]
df.iloc[:,0:2] = df.iloc[:,0:2] * 0.127
baseline_Blade_data = df.to_numpy()#convert to a numpy array
t_values = baseline_Blade_data[:,0]
print(t_values)

[0.0201254 0.0252054 0.0315579 0.0379079 0.0442579 0.0506079 0.0569579
 0.0633079 0.0696579 0.0760079 0.0823586 0.0887148 0.0950769 0.1014424
 0.107804  0.1141792 0.1173761 0.1205864 0.1238169 0.1255    0.12625
 0.127    ]


### LHS sample

In [4]:
## 基准的控制点
control_points = np.array([0.0198272, 0.03303, 0.01493, 0.00697, 55.05, 19.97, 16.71, 12.54])

## 上下界
lower_bound = 0.7  # 70%（min）
upper_bound = 1.3  # 130%（max）
num_samples = 1000 # the num of curves
maxPoints = control_points * upper_bound
minPoints = control_points * lower_bound
control_points_limits = np.column_stack([minPoints, maxPoints])

# 定义 B-spline 节点
degree = 3
chord_knots = np.concatenate(([0] * degree, [0.3984874, 0.89904882], [1] * degree))
twist_knots = np.concatenate(([0] * degree, [0.2, 0.89904882], [1] * degree))

#sample
sampling = LHS(xlimits=control_points_limits)
control_points_list = sampling(num_samples)

#get the geometry_data_list
geometry_list = []
for i in range(len(control_points_list)):
    sample_control_points = control_points_list[i]
    #Fitting chord and twist curves using BSpline method
    chord_spline = BSpline(chord_knots, sample_control_points[0:4], degree)
    twist_spline = BSpline(twist_knots, sample_control_points[4:8], degree)
    chord = chord_spline(t_values / 0.127)
    twist = twist_spline(t_values / 0.127)
    twist[0:2] = baseline_Blade_data[0:2, 2]
    geometry_data = np.vstack((
        [t_values],  # pos
        [chord],  # chord
        [twist]   # twist
    )).T  # Converted to one attribute per column
    geometry_list.append(geometry_data)

### 保存结果

In [9]:
np.save("geometry_data.npy", np.array(geometry_list))
print(len(geometry_list))

1000
